In [4]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample2") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

# Устанавливаем таймауты и keep-alive как числа (без 's')
# Значения в секундах или миллисекундах (зависит от версии, обычно keepalivetime в сек)
hadoop_conf.set("fs.s3a.threads.keepalivetime", "60") 
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.attempts.maximum", "10")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "5000")
hadoop_conf.set("fs.s3a.readahead.range", "65536")

hadoop_conf.set("fs.s3a.multipart.purge.age", "86400")

hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")


:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5743882-0527-41b8-906e-e5f4aaf78fce;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [2]:
s3_path_yandex = "s3a://dev/avpalatov/yandex/"

# Чтение parquet-файла
df = spark.read.parquet(s3_path_yandex)
df.show(5) 

+----------+------------------+---------------+--------------------+--------------+-------------------+--------------------+---------+-----------+----------+-------------+--------------+----------+
| ym:s:date|ym:s:regionCountry|ym:s:regionCity|ym:s:browserLanguage|  ym:s:browser|ym:s:deviceCategory|ym:s:operatingSystem|ym:s:hour|ym:s:visits|ym:s:users|ym:s:newUsers|ym:s:pageviews| update_at|
+----------+------------------+---------------+--------------------+--------------+-------------------+--------------------+---------+-----------+----------+-------------+--------------+----------+
|2026-01-26|            Russia|         Moscow|                  ru| Google Chrome|                 PC|Windows 10 (и пос...|    18:00|        4.0|       4.0|          0.0|           5.0|2026-01-27|
|2026-01-26|            Russia|           NULL|                  ru| Google Chrome|                 PC|Windows 10 (и пос...|    11:00|        3.0|       3.0|          0.0|           5.0|2026-01-27|
|2026-01-2

In [3]:
renamed_df = (df
                .withColumnRenamed("ym:s:date", "date")
                .withColumnRenamed("ym:s:regionCountry", "country")
                .withColumnRenamed("ym:s:regionCity", "city")
                .withColumnRenamed("ym:s:browserLanguage", "language")
                .withColumnRenamed("ym:s:browser", "browser")
                .withColumnRenamed("ym:s:deviceCategory", "device")
                .withColumnRenamed("ym:s:operatingSystem", "os")
                .withColumnRenamed("ym:s:hour", "hour")
                .withColumnRenamed("ym:s:visits", "visits")
                .withColumnRenamed("ym:s:users", "users")
                .withColumnRenamed("ym:s:newUsers", "new_users")
                .withColumnRenamed("ym:s:pageviews", "page_views")
)

renamed_df.show()

final_df = renamed_df 


+----------+-------+---------+--------+--------------+-----------+--------------------+-----+------+-----+---------+----------+----------+
|      date|country|     city|language|       browser|     device|                  os| hour|visits|users|new_users|page_views| update_at|
+----------+-------+---------+--------+--------------+-----------+--------------------+-----+------+-----+---------+----------+----------+
|2026-01-26| Russia|   Moscow|      ru| Google Chrome|         PC|Windows 10 (и пос...|18:00|   4.0|  4.0|      0.0|       5.0|2026-01-27|
|2026-01-26| Russia|     NULL|      ru| Google Chrome|         PC|Windows 10 (и пос...|11:00|   3.0|  3.0|      0.0|       5.0|2026-01-27|
|2026-01-26| Russia|   Moscow|      ru| Google Chrome|         PC|          Windows 11|13:00|   3.0|  3.0|      0.0|       4.0|2026-01-27|
|2026-01-26| Russia|   Moscow|      ru| Google Chrome|         PC|          Windows 11|14:00|   3.0|  2.0|      0.0|       3.0|2026-01-27|
|2026-01-26| Russia|   Mosc

In [ ]:
# ⬇️ Параметры подключения к CLICKHOUSE
jdbc_url = 'jdbc:clickhouse://clickhouse01:8123/avpalatov'
db_user = os.getenv('CLICKHOUSE_USER')
db_password = os.getenv('CLICKHOUSE_PASSWORD')
table_name = 'yandex_metrics'

final_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("truncate", "true") \
    .mode("append") \
    .save()

print("Таблица сохранена в Clickhouse")

In [ ]:
spark.stop()

In [3]:
import os
print(os.getenv("MINIO_PROD_BUCKET_NAME"))

prod
